### **Load and verify data**

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from scipy import stats

# Load the cleaned sales data from the CSV file
df = pd.read_csv("../data/processed/sales_data_cleaned_excel_native.csv",
                 encoding="utf-8-sig")

# Display basic information about the DataFrame
print(df.shape)
print(df.dtypes)
print("Total Sales:", df["total_sales"].sum())
print("Unique Sales IDs:", df["sale_id"].nunique())
print("Unique Order IDs:", df["order_id"].nunique())

# Display the first few rows of the DataFrame
df.head(10)

(28755, 20)
order_id                int64
sale_id                 int64
sale_date              object
order_code             object
transaction_type       object
sale_type              object
sales_channel          object
product_type_raw       object
product                object
net_quantity            int64
gross_sales           float64
discounts             float64
returns                 int64
net_sales             float64
shipping                int64
taxes                 float64
total_sales           float64
product_clean          object
product_type_clean     object
month                  object
dtype: object
Total Sales: 82792157.4
Unique Sales IDs: 28755
Unique Order IDs: 13257


,order_id,sale_id,sale_date,order_code,transaction_type,sale_type,sales_channel,product_type_raw,product,net_quantity,gross_sales,discounts,returns,net_sales,shipping,taxes,total_sales,product_clean,product_type_clean,month
0,2854884835480,8725409595544,2020-11-11T01:39:33+03:00,TS1011,product,order,Online Store,Bathing & Skin Care,Mamia Sensitive Bathing & Skin Care 64 Bathing...,1,200.0,0.0,0,200.0,0,0.0,200.0,Mamia Sensitive Bathing & Skin Care 64 Bathing...,Bathing & Skin Care,2020-11
1,2864488448152,8752046604440,2020-11-16T16:43:01+03:00,TS1012,product,order,Point of Sale,Baby Formula,NANNYcare Growing Up Milk Goat Milk Based 3 Fr...,2,9000.0,0.0,0,9000.0,0,0.0,9000.0,NANNYcare Growing Up Milk Goat Milk Based 3 Fr...,Baby Formula,2020-11
2,2864491954328,8752055845016,2020-11-16T16:45:58+03:00,TS1013,product,order,Point of Sale,Baby Diapers,DryNites Spider-Man Baby & Toddler Clothing 16pk,1,1800.0,0.0,0,1800.0,0,0.0,1800.0,DryNites Spider-Man Baby & Toddler Clothing 16pk,Baby Diapers,2020-11
3,2864491954328,8752055877784,2020-11-16T16:45:58+03:00,TS1013,product,order,Point of Sale,Bathing & Skin Care,Metanium Nappy Rash Ointment 30g,1,1500.0,0.0,0,1500.0,0,0.0,1500.0,Metanium Nappy Rash Ointment 30g,Bathing & Skin Care,2020-11
4,2864497066136,8752070656152,2020-11-16T16:49:47+03:00,TS1014,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,3000.0,0.0,0,3000.0,0,0.0,3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11
5,2864497066136,8752078684312,2020-11-16T16:52:36+03:00,TS1014,product,return,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,-1,0.0,0.0,-3000,-3000.0,0,0.0,-3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11
6,2864504438936,8752091201688,2020-11-16T16:56:04+03:00,TS1015,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,3000.0,0.0,0,3000.0,0,0.0,3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11
7,2864504438936,8752091234456,2020-11-16T16:56:04+03:00,TS1015,product,order,Point of Sale,NaN,Rider,1,300.0,0.0,0,300.0,0,0.0,300.0,Rider,Delivery Fee,2020-11
8,2864509321368,8752103653528,2020-11-16T16:59:30+03:00,TS1016,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,3000.0,0.0,0,3000.0,0,0.0,3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11
9,2864509321368,8752103620760,2020-11-16T16:59:30+03:00,TS1016,product,order,Point of Sale,Feeding and weaning,Joie Mimzy Snacker Highchair-Twinkle Linen,1,20000.0,0.0,0,20000.0,0,0.0,20000.0,Joie Mimzy Snacker Highchair-Twinkle Linen,Feeding And Weaning,2020-11


### **Mann-Whitney and cross check**
##### *Compare with T-test*

In [3]:
# Perform statistical analysis on sales data by sales channel
orders = (df.groupby(["order_id", "sales_channel"])["total_sales"]
            .sum().reset_index(name="order_total"))
orders = orders[orders["sales_channel"].isin(["Online Store", "Point of Sale"])]
online = orders.loc[orders["sales_channel"] == "Online Store", "order_total"]
pos    = orders.loc[orders["sales_channel"] == "Point of Sale", "order_total"]
print("Online_orders:", len(online), "Point of Sale_orders:", len(pos))

# Perform t-test and Mann-Whitney U test to compare sales between channels
t, p_t = stats.ttest_ind(online, pos, equal_var=False)
u, p_u = stats.mannwhitneyu(online, pos, alternative="two-sided")
print("T-Test p-value:", p_t)
print("Mann-Whitney U p-value:", p_u)

Online_orders: 996 Point of Sale_orders: 12260
T-Test p-value: 0.8247515734295636
Mann-Whitney U p-value: 0.2272352814138987


### **Build the Product table**

In [7]:
# Fee lines aren't products
prod = df[~df["product_type_clean"].isin(["Delivery Fee", "Shipping & Fees"])].copy()
prod["sale_day"] = pd.to_datetime(prod["sale_date"].str[:10])
prod["product_key"] = prod["product_clean"].str.lower().str.strip()

snapshot = prod["sale_day"].max() + pd.Timedelta(days=1)

rfm = (prod.groupby("product_key")
          .agg(recency=("sale_day", lambda s: (snapshot - s.max()).days),
               frequency=("order_id", "nunique"),
               monetary=("total_sales", "sum"))
          .reset_index())

print(len(prod), prod["product_clean"].nunique(), prod["product_key"].nunique(), snapshot.date())
print(rfm.describe().round(1))
prod.head(10)

21159 1799 1790 2023-11-01
       recency  frequency   monetary
count   1790.0     1790.0     1790.0
mean     402.2       11.6    45037.2
std      310.1       37.3   228816.5
min        1.0        1.0        0.0
25%      138.2        1.0     3000.0
50%      351.0        3.0     7450.0
75%      617.5        8.0    20000.0
max     1079.0      710.0  3963900.0


,order_id,sale_id,sale_date,order_code,transaction_type,sale_type,sales_channel,product_type_raw,product,net_quantity,...,returns,net_sales,shipping,taxes,total_sales,product_clean,product_type_clean,month,sale_day,product_key
0,2854884835480,8725409595544,2020-11-11T01:39:33+03:00,TS1011,product,order,Online Store,Bathing & Skin Care,Mamia Sensitive Bathing & Skin Care 64 Bathing...,1,...,0,200.0,0,0.0,200.0,Mamia Sensitive Bathing & Skin Care 64 Bathing...,Bathing & Skin Care,2020-11,2020-11-11,mamia sensitive bathing & skin care 64 bathing...
1,2864488448152,8752046604440,2020-11-16T16:43:01+03:00,TS1012,product,order,Point of Sale,Baby Formula,NANNYcare Growing Up Milk Goat Milk Based 3 Fr...,2,...,0,9000.0,0,0.0,9000.0,NANNYcare Growing Up Milk Goat Milk Based 3 Fr...,Baby Formula,2020-11,2020-11-16,nannycare growing up milk goat milk based 3 fr...
2,2864491954328,8752055845016,2020-11-16T16:45:58+03:00,TS1013,product,order,Point of Sale,Baby Diapers,DryNites Spider-Man Baby & Toddler Clothing 16pk,1,...,0,1800.0,0,0.0,1800.0,DryNites Spider-Man Baby & Toddler Clothing 16pk,Baby Diapers,2020-11,2020-11-16,drynites spider-man baby & toddler clothing 16pk
3,2864491954328,8752055877784,2020-11-16T16:45:58+03:00,TS1013,product,order,Point of Sale,Bathing & Skin Care,Metanium Nappy Rash Ointment 30g,1,...,0,1500.0,0,0.0,1500.0,Metanium Nappy Rash Ointment 30g,Bathing & Skin Care,2020-11,2020-11-16,metanium nappy rash ointment 30g
4,2864497066136,8752070656152,2020-11-16T16:49:47+03:00,TS1014,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,...,0,3000.0,0,0.0,3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11,2020-11-16,aptamil 1 first baby milk formula from birth 800g
5,2864497066136,8752078684312,2020-11-16T16:52:36+03:00,TS1014,product,return,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,-1,...,-3000,-3000.0,0,0.0,-3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11,2020-11-16,aptamil 1 first baby milk formula from birth 800g
6,2864504438936,8752091201688,2020-11-16T16:56:04+03:00,TS1015,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,...,0,3000.0,0,0.0,3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11,2020-11-16,aptamil 1 first baby milk formula from birth 800g
8,2864509321368,8752103653528,2020-11-16T16:59:30+03:00,TS1016,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,...,0,3000.0,0,0.0,3000.0,Aptamil 1 First Baby Milk Formula from Birth 800g,Baby Formula,2020-11,2020-11-16,aptamil 1 first baby milk formula from birth 800g
9,2864509321368,8752103620760,2020-11-16T16:59:30+03:00,TS1016,product,order,Point of Sale,Feeding and weaning,Joie Mimzy Snacker Highchair-Twinkle Linen,1,...,0,20000.0,0,0.0,20000.0,Joie Mimzy Snacker Highchair-Twinkle Linen,Feeding And Weaning,2020-11,2020-11-16,joie mimzy snacker highchair-twinkle linen
10,2864509321368,8752103686296,2020-11-16T16:59:30+03:00,TS1016,product,order,Point of Sale,Baby Formula,Aptamil 2 Follow On Milk Powder 800G,6,...,0,18000.0,0,0.0,18000.0,Aptamil 2 Follow On Milk Powder 800G,Baby Formula,2020-11,2020-11-16,aptamil 2 follow on milk powder 800g


### **Prepare and choose K**

In [8]:
# import necessary libraries for clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Prepare data for clustering
rfm["log_frequency"] = np.log1p(rfm["frequency"])
rfm["log_monetary"]  = np.log1p(rfm["monetary"])
X = StandardScaler().fit_transform(rfm[["recency", "log_frequency", "log_monetary"]])

# Perform KMeans clustering for different values of k and evaluate the results
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    print(f"Num Clusters: {k}, Inertia: {round(km.inertia_, 1)}, Silhouette Score: {round(silhouette_score(X, km.labels_), 3)}")

Num Clusters: 2, Inertia: 3157.2, Silhouette Score: 0.392
Num Clusters: 3, Inertia: 2173.8, Silhouette Score: 0.361
Num Clusters: 4, Inertia: 1728.8, Silhouette Score: 0.328
Num Clusters: 5, Inertia: 1468.2, Silhouette Score: 0.317
Num Clusters: 6, Inertia: 1264.9, Silhouette Score: 0.32
Num Clusters: 7, Inertia: 1101.3, Silhouette Score: 0.312
Num Clusters: 8, Inertia: 985.6, Silhouette Score: 0.289


### **Fit k=4 and profile the clusters**

In [ ]:
# Assign clusters to the RFM DataFrame using KMeans with 4 clusters
rfm["cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)

# Create a summary of the clusters with relevant metrics
summary = (rfm.groupby("cluster")
              .agg(products=("product_key", "count"),
                   avg_recency=("recency", "mean"),
                   avg_frequency=("frequency", "mean"),
                   avg_monetary=("monetary", "mean"),
                   total_revenue=("monetary", "sum")).round(1))
summary["pct_products"] = (summary["products"] / summary["products"].sum() * 100).round(1)
summary["pct_revenue"]  = (summary["total_revenue"] / summary["total_revenue"].sum() * 100).round(1)
print(summary.sort_values("avg_monetary", ascending=False))

         products  avg_recency  avg_frequency  avg_monetary  total_revenue  \
cluster                                                                      
3             261        198.0           57.8      253086.3     66055525.8   
1             636        197.1            5.7       14300.9      9095395.0   
2             436        838.7            3.2        9963.1      4343895.6   
0             457        387.9            1.5        2454.6      1121732.0   

         pct_products  pct_revenue  
cluster                             
3                14.6         81.9  
1                35.5         11.3  
2                24.4          5.4  
0                25.5          1.4  
